# Topic 7: Advanced RAG and Web Search

Welcome to Day 3. Yesterday we built a vector RAG pipeline that answers questions
from Barclays product PDFs. Today we hit the wall every production RAG team hits:
**the docs go stale**.

A customer asks "what is the current Personal Loan rate today?" Our vector store
says 6.5%. The barclays.co.uk page says 6.3%. The vector store is wrong, and
re-embedding the whole corpus every time a rate changes is not a real strategy.

In this notebook we add a second retrieval path: live web search restricted to
barclays.co.uk via the OpenAI Responses API. We then build a router that decides
per-query whether to use the vector store, the web tool, or both, with a safe
fallback when the web call fails.

## What you will build

By the end of this notebook you will have:
1. A `web_search_barclays(query)` helper that calls the Responses API web_search
   tool with an `allowed_domains` filter so we never leave barclays.co.uk
2. An `extract_citations(response)` helper that pulls the `url_citation`
   annotations out of the Responses API output so every answer is auditable
3. A `route_query(query, collection)` decision function that picks vector,
   web, or hybrid based on vector confidence and query intent
4. A production-grade `hybrid_answer(query, collection)` that wires it all
   together with a try/except fallback to vector-only when web search fails

## Why this matters at Barclays

The FCA Consumer Duty (PS22/9, in force since July 2023) requires customer-facing
AI to be grounded in verified source material. A chatbot that hallucinates a
mortgage rate is a regulatory incident, not a UX bug. Hybrid RAG with explicit
URL citations is the FCA-aligned pattern: the vector store keeps you fast on
known docs, web search keeps you fresh, and citations give you the audit trail.

We also stay strictly inside the information-only boundary. Quoting "the current
Personal Loan APR is 6.3%" from barclays.co.uk is information. Telling a
specific customer "you should take this loan" is regulated advice. Topic 8 will
add the output guardrail that enforces this boundary; today we lay the
factual-sourcing groundwork it relies on.

## Prerequisites

You completed Topic 6 (RAG Foundations). You have an OpenAI API key. The
ChromaDB store from Topic 6 may still be on disk at `./chroma_db` - we will
reuse it if it is, or rebuild it in 2 seconds if not.

## Section 0: Environment setup

Same drill as Topic 6: install pinned versions, load the OpenAI key with
`getpass`, and rebuild the helpers we will lean on. Skim this section, run all
cells, and we will be at the new material in 5 minutes.

In [ ]:
# openai>=2.30.0 is required for the Responses API web_search tool with allowed_domains filter
# chromadb==1.5.8 matches the Topic 6 pin
# pymupdf 1.27.2.2 is the canonical PDF text extractor (same pin as Topic 2).
# requests is used to download the PDFs over HTTPS from the public S3 bucket.
# sagemaker==2.232.1 is the last classic SDK release where get_execution_role lives at the top level
# boto3 is the AWS SDK - needed for S3 and SageMaker session
# numpy<2 is mandatory: chromadb 1.x breaks on numpy 2.x
!pip install -q "chromadb==1.5.8" "openai>=2.30.0" "pymupdf==1.27.2.2" "requests" "sagemaker==2.232.1" "boto3" "numpy<2"

In [ ]:
import os
import json
import time
import getpass

from openai import OpenAI
import chromadb

# Load the OpenAI key from getpass if it is not already in the environment
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

client = OpenAI()
print("OpenAI client ready")

## What are we building today?

Today we extend the Barclays Product Knowledge Assistant from Topic 6 with a
**hybrid retrieval layer**:

```
Customer query
    |
    v
+-------------------+
| route_query       |  <-- decides per-query
+-------------------+
    |
    +--> vector only      (high vector confidence, no rate keywords)
    +--> web only         (low vector confidence)
    +--> hybrid           (rate / current / today keywords)
    |
    v
+-------------------+
| hybrid_answer     |  <-- merges results, calls gpt-4o, returns answer + citations
+-------------------+
    |
    v
"Personal Loan APR is currently 6.3% (source: barclays.co.uk/loans/personal-loan)"
```

The three concepts map to three labs:
1. **Lab 1 (Tier 1)**: Build `web_search_barclays(query)` using the Responses
   API with domain restriction
2. **Lab 2 (Tier 1)**: Build `extract_citations(response)` to parse URL
   citations from the Responses API output
3. **Lab 3 (Tier 2 HARD)**: Build `hybrid_answer(query, collection)` that
   routes the query, calls the right retrieval path, falls back gracefully
   when web search fails, and returns answer + merged citations

By the end you will have the hybrid retrieval module that the Day 3 capstone
imports as `policy_lookup`.

In [ ]:
# Filesystem-only handoff: every notebook downloads its corpus fresh from S3.
# We do NOT carry Python globals from prior notebooks. Each notebook is self-contained.
# This is the same pattern Topic 2 and Topic 6 use: HTTPS download via requests,
# PyMuPDF extract, regex clean, fixed-size chunk with overlap.
import requests
import pymupdf
import re
from io import BytesIO

# Public S3 bucket - no IAM credentials needed. Same files as Topics 2 and 6.
S3_BUCKET = "barclays-prompt-eng-data"
S3_REGION = "us-east-2"
S3_BASE_URL = f"https://{S3_BUCKET}.s3.{S3_REGION}.amazonaws.com"

DOWNLOAD_HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Accept": "application/pdf,*/*",
}

# Two PDFs - keeps the corpus diverse and the download under 5 seconds.
PDF_KEYS = [
    "barclays_personal_loan_faq.pdf",
    "barclays_credit_card_tnc.pdf",
]


def _load_pdf_from_s3(key: str) -> str:
    """Download a PDF over HTTPS and return its full text. Raises on failure."""
    url = f"{S3_BASE_URL}/{key}"
    resp = requests.get(url, headers=DOWNLOAD_HEADERS, timeout=30)
    resp.raise_for_status()
    if not resp.content:
        raise RuntimeError(f"Empty PDF body for {key}")
    doc = pymupdf.open(stream=BytesIO(resp.content), filetype="pdf")
    pages = [page.get_text("text") for page in doc]
    doc.close()
    return "\n\n".join(pages)


def _clean_pdf_text(text: str) -> str:
    """Same cleaning pipeline as Topic 2 Cell afb66ad6."""
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)
    text = re.sub(r"Barclays Bank UK PLC\s+Page\s+\d+\s+of\s+\d+[^\n]*\n", "", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"^\s*[-_.=*]{3,}\s*$", "", text, flags=re.MULTILINE)
    text = re.sub(r"^\s*\d+\s*$", "", text, flags=re.MULTILINE)
    return text.strip()


def _chunk_text(text: str, chunk_size: int = 1500, overlap: int = 200) -> list:
    """Same chunker as Topic 2 Cell 258f7171. Sentence-boundary aware."""
    if not text:
        return []
    out, start = [], 0
    while start < len(text):
        end = start + chunk_size
        if end < len(text):
            boundary = max(
                text.rfind(". ", start, end),
                text.rfind(".\n", start, end),
                text.rfind("\n\n", start, end),
            )
            if boundary > start + chunk_size // 2:
                end = boundary + 1
        chunk = text[start:end].strip()
        if chunk:
            out.append(chunk)
        start += chunk_size - overlap
    return out


# Download, clean, chunk - one PDF at a time.
BARCLAYS_DOCS = []
download_errors = []
for key in PDF_KEYS:
    try:
        raw = _load_pdf_from_s3(key)
        cleaned = _clean_pdf_text(raw)
        new_chunks = _chunk_text(cleaned)
        BARCLAYS_DOCS.extend(new_chunks)
        print(f"[OK]   {key}: {len(raw):,} chars raw -> {len(cleaned):,} chars cleaned -> {len(new_chunks)} chunks")
    except Exception as e:
        download_errors.append((key, str(e)))
        print(f"[FAIL] {key}: {e}")

# Fallback only if S3 was completely unreachable.
if not BARCLAYS_DOCS:
    print()
    print("!!! WARNING: S3 download failed for ALL PDFs. Using small inline fallback corpus.")
    print("!!! Check that the S3 bucket is reachable and PDFs are uploaded.")
    print("!!! Errors:", download_errors)
    BARCLAYS_DOCS = [
        "Barclays Personal Loan: borrow 1,000 GBP to 35,000 GBP. Representative APR 6.5 percent.",
        "Barclays Rewards Credit Card: 0.25 percent cashback. Foreign transaction fee 0 percent.",
        "Barclays Savings Account: instant access, variable rate currently 3.75 percent AER.",
    ]


# Topic 6 helpers, rebuilt inline so this notebook is self-contained
def embed_texts(texts):
    """Return a list of 1536-dim embeddings for the given texts."""
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=texts
    )
    return [item.embedding for item in response.data]


def add_to_store(collection, docs, embeddings):
    """Insert docs and their embeddings into the collection with sequential ids."""
    ids = [f"doc_{i}" for i in range(collection.count(), collection.count() + len(docs))]
    collection.add(ids=ids, documents=docs, embeddings=embeddings)


def retrieve(query, collection, n_results=3):
    """Return the top-n document strings most similar to the query."""
    query_embedding = embed_texts([query])[0]
    results = collection.query(query_embeddings=[query_embedding], n_results=n_results)
    return results["documents"][0]


# Reuse the on-disk ChromaDB if Topic 6 left one behind, otherwise create a fresh one.
chroma_client = chromadb.PersistentClient(path="./chroma_db")
collection = chroma_client.get_or_create_collection(
    name="barclays_products",
    configuration={"hnsw": {"space": "cosine"}}
)

# If the collection is empty, embed the freshly-downloaded chunks (takes a few seconds).
if collection.count() == 0:
    print()
    print(f"Empty collection, embedding {len(BARCLAYS_DOCS)} chunks...")
    embeddings = embed_texts(BARCLAYS_DOCS)
    add_to_store(collection, BARCLAYS_DOCS, embeddings)

print()
print(f"Collection ready with {collection.count()} documents")

## Concept 1: The vector store goes stale

A customer asks: "what is the current Personal Loan APR at Barclays today?"

Our Topic 6 pipeline confidently answers using whatever was in the PDF when we
ingested it. That worked yesterday. Watch what happens when the underlying
product page changes faster than our re-embedding cadence.

### Beat 1: see the problem

Run the next cell. We use the Topic 6 `retrieve` and a plain `client.responses.create`
to answer the "current rate" question from the vector store only.

In [ ]:
# A customer-facing question that needs CURRENT data
query = "What is the current Personal Loan APR at Barclays today?"

# Pull the top-2 chunks from the vector store
chunks = retrieve(query, collection, n_results=2)

# Build the standard Topic 6 RAG prompt
context = "\n\n".join(chunks)
prompt = (
    "Answer the customer's question using only the Barclays product context provided.\n\n"
    f"Context:\n{context}\n\n"
    f"Customer question: {query}"
)

# Vector-only answer using the Responses API
response = client.responses.create(
    model="gpt-4o",
    input=prompt
)
print("Vector-only answer:")
print(response.output_text)
print()
print("Problem: this number came out of our static vector store. "
      "If the product page on barclays.co.uk has been updated since we ingested it, "
      "we just told the customer something stale, with no way for them to verify it.")

### Beat 2: the fix

We need a second retrieval path that pulls live from the Barclays website.
The OpenAI Responses API has a built-in `web_search` tool that does exactly
this and, critically, it accepts an `allowed_domains` filter so we never
leave barclays.co.uk.

```mermaid
graph LR
    subgraph Vector["Vector Store (ChromaDB)"]
        V1[Query embed] --> V2[Cosine search]
        V2 --> V3["APR 6.5% - STALE"]
    end
    subgraph Web["Web Search (barclays.co.uk)"]
        W1[Query] --> W2[barclays.co.uk]
        W2 --> W3["APR 6.3% - LIVE"]
    end
    V3 --> LLM[gpt-4o]
    W3 --> LLM
    LLM --> Answer[Grounded answer]
```

Two non-obvious points before we run the demo:

1. **You must use the Responses API**, not Chat Completions. The
   `filters.allowed_domains` parameter is a Responses API feature. The
   `client.chat.completions.create` route does not support it.

2. **Always cap tool calls.** Pass `max_tool_calls=2` or similar. Without a
   cap the model can enter an infinite tool-calling loop that runs up your
   bill while you wait for a timeout.

### Beat 3: live demo

In [ ]:
def web_search_barclays_demo(query):
    """Demo version of the live web search call - we will productionise this in Lab 1."""
    start = time.time()
    # The web_search tool is built into the Responses API
    # filters.allowed_domains keeps the model on barclays.co.uk only
    # max_tool_calls=2 prevents runaway tool-calling loops
    response = client.responses.create(
        model="gpt-4o",
        tools=[{
            "type": "web_search",
            "filters": {"allowed_domains": ["barclays.co.uk"]}
        }],
        input=query,
        max_tool_calls=2,
    )
    latency = time.time() - start
    return response, latency

response, latency = web_search_barclays_demo(
    "What is the current Personal Loan representative APR at Barclays UK?"
)
print(f"Web search call took {latency:.1f} seconds")
print()
print("Answer text:")
print(response.output_text)
print()
print("Output items returned by the API:")
for item in response.output:
    # Note both web_search_call and message items in the output stream
    print(f"  - type={item.type}")

### Beat 4: Lab 1 (Tier 1, ~15 min)

Wrap the demo in a clean function the rest of the notebook will call.

**Your task**: implement `web_search_barclays(query)` that:
1. Calls `client.responses.create` with `model="gpt-4o"`
2. Passes the `web_search` tool with `allowed_domains=["barclays.co.uk"]`
3. Caps tool calls at 2 with `max_tool_calls=2`
4. Measures wall-clock latency in seconds
5. Returns a dict with three keys:
   - `"text"`: the `response.output_text` string
   - `"response"`: the raw response object (Lab 2 needs the annotations)
   - `"latency_s"`: latency rounded to 1 decimal

**Stretch (if you finish early)**: wrap the API call in a try/except that
catches any exception, prints a one-line warning, and returns
`{"text": "", "response": None, "latency_s": -1.0, "error": str(e)}`. The
fallback path in Lab 3 uses this signal.

**Homework extension**: add an exponential backoff retry using `tenacity`
(`pip install tenacity` outside the notebook). Retry on the OpenAI rate-limit
exception only, with 3 attempts and a 2-second base wait.

In [ ]:
# Lab 1 SOLUTION: web_search_barclays
def web_search_barclays(query):
    """Call the OpenAI Responses API with web_search restricted to barclays.co.uk.

    Returns:
        dict with keys 'text', 'response', 'latency_s'.
    """
    # SOLUTION: measure start time
    start = time.time()

    # SOLUTION: call client.responses.create with the web_search tool restricted to barclays.co.uk
    # max_tool_calls=2 prevents runaway tool-calling loops
    response = client.responses.create(
        model="gpt-4o",
        tools=[{
            "type": "web_search",
            "filters": {"allowed_domains": ["barclays.co.uk"]}
        }],
        input=query,
        max_tool_calls=2,
    )

    # SOLUTION: compute latency in seconds
    latency = time.time() - start

    # SOLUTION: return the dict with text, raw response (Lab 2 needs annotations), and latency
    return {
        "text": response.output_text,
        "response": response,
        "latency_s": round(latency, 1),
    }


result = web_search_barclays("What is the current Personal Loan APR at Barclays UK?")
print(f"Latency: {result['latency_s']}s")
print(f"Text: {result['text'][:300]}...")

In [ ]:
# Safety-net: if your Lab 1 implementation is incomplete, run this cell to get a
# working web_search_barclays so you can continue with Lab 2 and Lab 3.
if 'result' not in dir() or not isinstance(result, dict) or 'response' not in result:
    def web_search_barclays(query):
        start = time.time()
        response = client.responses.create(
            model="gpt-4o",
            tools=[{
                "type": "web_search",
                "filters": {"allowed_domains": ["barclays.co.uk"]}
            }],
            input=query,
            max_tool_calls=2,
        )
        return {
            "text": response.output_text,
            "response": response,
            "latency_s": round(time.time() - start, 1),
        }
    result = web_search_barclays("What is the current Personal Loan APR at Barclays UK?")
    print(f"Safety-net executed. Latency: {result['latency_s']}s")
else:
    print("Lab 1 implementation looks good, continuing with student version.")

## Concept 2: Source attribution and citations

Lab 1 gave us a fresh answer. But look at what we returned: a string of text
with no machine-readable trail back to the source page. In a regulated context
that is a non-starter.

The Responses API actually does emit citation data, it just hides it inside
the `response.output[].content[].annotations` structure. We need to pull
those annotations out into a clean list we can show to the customer and log
for the audit trail.

### Beat 1: see the problem

In [ ]:
text_only = result["text"]
print("What we have right now:")
print(text_only[:400])
print()
print("Problem: a customer reading this has no way to verify the number, and")
print("an FCA auditor has no way to trace it back to a barclays.co.uk page.")
print("We need to surface the URL annotations the API already returned.")

### Beat 2: the fix

The Responses API returns a list of output items. Each `message` item has a
`content` list. Each `content` item has a `text` field AND an `annotations`
field. The annotations include `url_citation` entries with `url` and `title`.

```mermaid
sequenceDiagram
    participant C as Caller
    participant API as Responses API
    participant Web as barclays.co.uk
    C->>API: responses.create(tools=[web_search], input=query)
    API->>Web: web_search_call(query, allowed_domains)
    Web-->>API: search results
    API-->>C: response.output[]
    Note over C: output[0].type = web_search_call
    Note over C: output[1].type = message
    Note over C: message.content[0].text = answer
    Note over C: message.content[0].annotations = url_citations
```

We walk that structure once and collect the citations into a deduplicated list.

### Beat 3: live demo

In [ ]:
def extract_citations_demo(response):
    """Demo version: walk the Responses output and collect url_citation annotations."""
    citations = []
    seen_urls = set()
    # Iterate the top-level output items
    for item in response.output:
        # Only message items carry annotated text content
        if item.type != "message":
            continue
        for content_item in item.content:
            # annotations may be missing on some content items, default to empty
            annotations = getattr(content_item, "annotations", None) or []
            for ann in annotations:
                # We only care about URL citations here
                if getattr(ann, "type", None) != "url_citation":
                    continue
                url = ann.url
                # Dedupe by url so a customer does not see the same source twice
                if url in seen_urls:
                    continue
                seen_urls.add(url)
                citations.append({
                    "url": url,
                    "title": getattr(ann, "title", "") or "",
                })
    return citations

demo_citations = extract_citations_demo(result["response"])
print(f"Found {len(demo_citations)} unique citations:")
for c in demo_citations:
    print(f"  - {c['title'] or '(no title)'}: {c['url']}")

### Beat 4: Lab 2 (Tier 1, ~15 min)

Now write the production version. Same idea, same shape, but with two extra
guarantees the demo skipped.

**Your task**: implement `extract_citations(response)` that:
1. Iterates `response.output` looking for items with `type == "message"`
2. For each message item, walks `content` and pulls every annotation
3. Keeps only annotations whose `type == "url_citation"`
4. Deduplicates by `url`
5. Returns a list of dicts with keys `"url"` and `"title"`. If `title` is
   missing or empty, fall back to the URL itself as the title (so there is
   always something to display)
6. Handles the case where `response.output` is empty (returns `[]`)
7. Handles the case where a content item has no `annotations` attribute
   (skips it, does not raise)

**Stretch (if you finish early)**: also extract the `start_index` and
`end_index` fields from the annotation if present, and return them in each
dict so a UI could highlight which span of the answer text the citation
supports.

**Homework extension**: extend to handle the case where the same URL appears
with different titles in different annotations - keep the longest non-empty
title.

In [ ]:
# Lab 2 SOLUTION: extract_citations
def extract_citations(response):
    """Extract URL citations from a Responses API response.

    Returns:
        list of {"url": str, "title": str} dicts, deduplicated by url.
    """
    citations = []
    seen_urls = set()

    # SOLUTION: handle the empty response.output case defensively
    if not response or not getattr(response, "output", None):
        return citations

    # SOLUTION: iterate response.output, keep only items with type == 'message'
    for item in response.output:
        if item.type != "message":
            continue

        # SOLUTION: iterate content list inside each message item
        for content_item in item.content:

            # SOLUTION: get annotations safely (may be missing on some content items)
            annotations = getattr(content_item, "annotations", None) or []

            # SOLUTION: keep only annotations with type == 'url_citation'
            for ann in annotations:
                if getattr(ann, "type", None) != "url_citation":
                    continue

                # SOLUTION: dedupe by url, fall back to url as title if title empty
                url = ann.url
                if url in seen_urls:
                    continue
                seen_urls.add(url)
                title = getattr(ann, "title", "") or url
                citations.append({"url": url, "title": title})

    return citations


citations = extract_citations(result["response"])
print(f"Found {len(citations)} citations:")
for c in citations:
    print(f"  - {c['title']}: {c['url']}")

In [ ]:
# Safety-net: if your Lab 2 implementation is incomplete, run this cell to get a
# working extract_citations so you can continue with Lab 3.
if 'citations' not in dir() or not isinstance(citations, list):
    def extract_citations(response):
        citations = []
        seen_urls = set()
        if not response or not getattr(response, "output", None):
            return citations
        for item in response.output:
            if item.type != "message":
                continue
            for content_item in item.content:
                annotations = getattr(content_item, "annotations", None) or []
                for ann in annotations:
                    if getattr(ann, "type", None) != "url_citation":
                        continue
                    url = ann.url
                    if url in seen_urls:
                        continue
                    seen_urls.add(url)
                    title = getattr(ann, "title", "") or url
                    citations.append({"url": url, "title": title})
        return citations
    citations = extract_citations(result["response"])
    print(f"Safety-net executed. Found {len(citations)} citations.")
else:
    print("Lab 2 implementation looks good, continuing with student version.")

## Concept 3: Hybrid routing with safe fallback

We can now do vector RAG (Topic 6) and we can do live web search (Concept 1
and 2). The naive option is to always do both. That is wrong for two reasons:

1. **Cost and latency**: a web_search call adds 2 to 5 seconds of latency and
   real money per query. Running it on "what is your address?" is wasteful.
2. **Reliability**: web search can fail (rate limits, transient errors,
   barclays.co.uk being slow). If we always require it, the assistant goes
   down whenever the web tool does.

The production pattern is a **router** that decides per-query, plus a
**fallback** that degrades gracefully to vector-only when the web call fails.

### Beat 1: see the problem

In [ ]:
def naive_always_web(query, collection):
    """The naive approach: always call both retrieval paths regardless of intent."""
    chunks = retrieve(query, collection, n_results=2)
    web = web_search_barclays(query)
    return {"vector_chunks": chunks, "web_text": web["text"], "web_latency_s": web["latency_s"]}

cheap_query = "What is the maximum amount I can borrow on a Barclays Personal Loan?"
print(f"Query: {cheap_query}")
print("(this is answerable from the vector store alone, web search adds no value)")
print()
out = naive_always_web(cheap_query, collection)
print(f"Web search added {out['web_latency_s']}s of latency for nothing.")
print("In a 1000-query/day deployment that is real money and real wait time.")

### Beat 2: the fix

We route per-query. The router has two signals:

1. **Vector confidence**: the cosine similarity of the top vector hit. If the
   vector store has a confident answer, we trust it.
2. **Intent keywords**: words like "current", "today", "rate", "promotion",
   "live" mean "this answer goes stale, please go live".

```mermaid
flowchart TD
    A[Customer query] --> B[embed + vector_confidence]
    B --> C{Rate keyword in query?}
    C -->|Yes + conf >= 0.5| D[HYBRID vector + web]
    C -->|Yes + conf < 0.5| E[WEB only]
    C -->|No| F{conf >= 0.75?}
    F -->|Yes| G[VECTOR only fast + cheap]
    F -->|No| E
    D --> H{Web call succeeds?}
    H -->|No| G
    E --> H
```

The fallback is a try/except around the web call. If it fails for any
reason, we log a warning, route the query to vector-only, and add a
synthetic citation that says "live data unavailable, answer based on cached
product documentation".

### Beat 3: live demo

Below is the demo version. Lab 3 asks you to harden it.

In [ ]:
# Words that signal "this answer goes stale, fetch live data"
RATE_KEYWORDS = ["current", "today", "live", "rate", "apr", "aer", "promotion", "offer", "now"]

def vector_confidence(query, collection, n_results=1):
    """Return cosine similarity of the top vector hit (1.0 = perfect, 0.0 = no signal)."""
    query_embedding = embed_texts([query])[0]
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results,
        include=["distances"],
    )
    distances = results["distances"][0]
    if not distances:
        return 0.0
    # ChromaDB cosine distance: similarity = 1 - distance
    similarity = 1.0 - distances[0]
    return similarity

def route_query(query, collection):
    """Pick the retrieval path: 'vector', 'web', or 'hybrid'."""
    has_rate_kw = any(kw in query.lower() for kw in RATE_KEYWORDS)
    confidence = vector_confidence(query, collection)
    # Rate keyword + we have related docs -> blend both
    if has_rate_kw and confidence >= 0.5:
        return "hybrid", f"rate keyword detected, vector confidence {confidence:.2f}"
    # Rate keyword but no related docs -> just go web
    if has_rate_kw and confidence < 0.5:
        return "web", f"rate keyword, vector confidence too low ({confidence:.2f})"
    # No rate keyword and good vector hit -> trust the vector store
    if confidence >= 0.75:
        return "vector", f"vector confidence high ({confidence:.2f})"
    # Fall through: weak vector signal and no rate keyword -> still go web for safety
    return "web", f"vector confidence low ({confidence:.2f}), no rate keyword"

def hybrid_answer_demo(query, collection):
    """Demo version (no fallback). Lab 3 productionises this with try/except."""
    route, reason = route_query(query, collection)
    print(f"[router] route={route}, reason={reason}")

    if route == "vector":
        chunks = retrieve(query, collection, n_results=3)
        prompt = (
            "You share factual Barclays product information only. "
            "You do not provide personalised financial advice.\n\n"
            f"Context:\n{chr(10).join(chunks)}\n\n"
            f"Customer question: {query}"
        )
        response = client.responses.create(model="gpt-4o", input=prompt)
        return {"answer": response.output_text, "citations": [{"url": "internal://barclays_products", "title": "Barclays product documentation"}], "route": route}

    web = web_search_barclays(query)
    web_citations = extract_citations(web["response"]) if web["response"] else []

    if route == "web":
        return {"answer": web["text"], "citations": web_citations, "route": route}

    # Hybrid: merge vector chunks and web text into one prompt
    chunks = retrieve(query, collection, n_results=2)
    merged_prompt = (
        "You share factual Barclays product information only. "
        "You do not provide personalised financial advice. "
        "Prefer the live web information for any rate or current figure.\n\n"
        f"Internal product context:\n{chr(10).join(chunks)}\n\n"
        f"Live web information:\n{web['text']}\n\n"
        f"Customer question: {query}"
    )
    response = client.responses.create(model="gpt-4o", input=merged_prompt)
    internal_cite = [{"url": "internal://barclays_products", "title": "Barclays product documentation"}]
    return {"answer": response.output_text, "citations": internal_cite + web_citations, "route": route}

demo_out = hybrid_answer_demo("What is the current Personal Loan APR at Barclays today?", collection)
print()
print("Answer:")
print(demo_out["answer"])
print()
print(f"Citations ({len(demo_out['citations'])}):")
for c in demo_out["citations"]:
    print(f"  - {c['title']}: {c['url']}")

### Beat 4: Lab 3 (Tier 2 HARD, ~25 min)

Now turn the demo into the production version Topic 9 will import. The demo
will crash if web_search_barclays raises. The production version must not.

**Your task**: implement `hybrid_answer(query, collection)` (yes, the same
name - we are replacing the demo with the real one) that:

1. Calls `route_query` to pick `"vector"`, `"web"`, or `"hybrid"`
2. For the `"vector"` route: behaves like Topic 6's `rag_answer` but returns
   the dict shape with `"answer"`, `"citations"`, `"route"`, `"fell_back"`
3. For the `"web"` and `"hybrid"` routes: wraps the `web_search_barclays`
   call in a try/except. On exception:
   - Print a one-line warning with the exception message
   - Set `fell_back=True`
   - Re-route to vector-only
   - Add a synthetic citation
     `{"url": "internal://fallback", "title": "Live data unavailable, answer from cached product documentation"}`
4. For the `"hybrid"` route happy-path: merge the vector chunks and the web
   text into one prompt, generate the final answer with gpt-4o, and return
   both the internal citation and the web citations
5. Always uses the system instruction "You share factual Barclays product
   information only. You do not provide personalised financial advice."
6. Returns a dict with keys `"answer"`, `"citations"`, `"route"`, `"fell_back"`

**Stretch (if you finish early)**: add a `"latency_s"` key to the return dict
that measures total wall-clock time. Print a warning if total latency
exceeds 8 seconds.

**Homework extension**: add semantic caching. Before calling `hybrid_answer`,
embed the query and check a small in-memory dict of (embedding, response)
pairs. If cosine similarity to any cached query is above 0.95, return the
cached response. Otherwise compute the answer and add it to the cache. This
is the single highest-ROI optimisation on production RAG systems.

Stretch data: load `barclays_chunks.json` via HTTP: `url = 'https://barclays-prompt-eng-data.s3.us-east-2.amazonaws.com/barclays_chunks.json'` then `chunks = requests.get(url).json()` if you want to test against real Barclays product text. See `setup/INSTRUCTOR_SETUP.ipynb` Cell 10 for the load snippet.

In [ ]:
# Lab 3 SOLUTION: hybrid_answer with safe fallback
def hybrid_answer(query, collection):
    """Production-grade hybrid retrieval: vector + web with safe fallback.

    Returns:
        dict with keys 'answer', 'citations', 'route', 'fell_back'.
    """
    SYSTEM = (
        "You share factual Barclays product information only. "
        "You do not provide personalised financial advice."
    )

    # SOLUTION: call route_query to decide the path
    route, reason = route_query(query, collection)
    fell_back = False

    # Helper for the vector-only branch (we may also use it as a fallback path)
    def _vector_branch(extra_citation=None):
        chunks = retrieve(query, collection, n_results=3)
        prompt = (
            f"{SYSTEM}\n\n"
            f"Context:\n{chr(10).join(chunks)}\n\n"
            f"Customer question: {query}"
        )
        response = client.responses.create(model="gpt-4o", input=prompt)
        citations = [{"url": "internal://barclays_products", "title": "Barclays product documentation"}]
        if extra_citation:
            citations.append(extra_citation)
        return response.output_text, citations

    # SOLUTION: handle the 'vector' route - simple vector RAG, no web call
    if route == "vector":
        answer, citations = _vector_branch()
        return {"answer": answer, "citations": citations, "route": route, "fell_back": False}

    # SOLUTION: for 'web' and 'hybrid' routes, wrap web_search_barclays in try/except
    try:
        web = web_search_barclays(query)
        web_citations = extract_citations(web["response"]) if web["response"] else []
    except Exception as e:
        # On exception: warn, set fell_back=True, re-route to vector-only,
        # and add a synthetic fallback citation so the audit trail is honest
        print(f"[warning] web_search_barclays failed: {e}. Falling back to vector-only.")
        fallback_cite = {"url": "internal://fallback", "title": "Live data unavailable, answer from cached product documentation"}
        answer, citations = _vector_branch(extra_citation=fallback_cite)
        return {"answer": answer, "citations": citations, "route": route, "fell_back": True}

    # SOLUTION: 'web' happy-path - return web text and extracted citations
    if route == "web":
        return {"answer": web["text"], "citations": web_citations, "route": route, "fell_back": False}

    # SOLUTION: 'hybrid' happy-path - merge vector chunks + web text into one prompt
    chunks = retrieve(query, collection, n_results=2)
    merged_prompt = (
        f"{SYSTEM} "
        "Prefer the live web information for any rate or current figure.\n\n"
        f"Internal product context:\n{chr(10).join(chunks)}\n\n"
        f"Live web information:\n{web['text']}\n\n"
        f"Customer question: {query}"
    )
    response = client.responses.create(model="gpt-4o", input=merged_prompt)
    internal_cite = [{"url": "internal://barclays_products", "title": "Barclays product documentation"}]
    return {
        "answer": response.output_text,
        "citations": internal_cite + web_citations,
        "route": route,
        "fell_back": False,
    }


for q in [
    "What is the maximum amount I can borrow on a Barclays Personal Loan?",
    "What is the current Personal Loan APR at Barclays today?",
    "Tell me about the Barclays Travel Wallet supported currencies.",
]:
    print(f"Q: {q}")
    out = hybrid_answer(q, collection)
    print(f"  route={out['route']}, fell_back={out['fell_back']}")
    print(f"  A: {out['answer'][:200]}...")
    print(f"  citations: {len(out['citations'])}")
    print()

## Wrap-up

You just built the hybrid retrieval layer of the Barclays Product Knowledge
Assistant. Three things you can do now that you could not at the start of
the session:

1. **Pull live facts from barclays.co.uk** with one function call, with a
   hard guarantee the model never wanders off-domain
2. **Hand the customer a citation** for every claim, with a clean dedup and
   audit-friendly URL list
3. **Route per-query** between vector, web, and hybrid with a graceful
   fallback when the web tool fails - the assistant keeps working even when
   the live tool is down

### Where this fits

- Topic 6 gave you `rag_answer` for static product docs.
- Topic 7 gives you `hybrid_answer` for static + live with fallback.
- Topic 8 (next, in 5 minutes) wraps `hybrid_answer` in input and output
  guardrails: PII detection on the way in, financial-advice boundary
  enforcement on the way out, escalation triggers when the customer needs a
  human.
- Topic 9 (capstone) imports `hybrid_answer` as the `policy_lookup` tool of
  the Transaction Query Agent.

### Production take-homes

A few things you would do before shipping this for real that we did not do
in the lab for time reasons:

- **Retries with exponential backoff**: wrap `client.responses.create` with
  `tenacity` retry on `RateLimitError` and `APITimeoutError`, 3 attempts,
  base wait 2 seconds. The OpenAI SDK retries by default but only on a small
  set of errors.
- **Semantic caching**: a 0.95 cosine similarity cache cuts cost and
  latency on repeat queries by 40 to 70% in typical chatbot traffic. The
  homework extension on Lab 3 walks through it.
- **Cost tracking**: log `response.usage.input_tokens` and
  `response.usage.output_tokens` per call. Web search adds a billable
  per-call charge separate from token cost - log it.
- **Latency budget**: if hybrid exceeds 8 seconds, fall back to vector-only.
  Customers abandon at 10 seconds.
- **Compliance**: log every (query, route, citations) tuple to a write-once
  store. The FCA expects a 6-year audit trail.

See you in Topic 8.